<span style="font-size: 5em">🦜</span>

# __LangGraph Essentials__
## Lab 5: Interrupt! Human In The Loop
<div style="display:flex; align-items:flex-start;">
  <img src="../assets/HITL.png" width="500" style="margin-right:15px;"/>
</div>

LangGraph 的 `interrupt()` 函数会暂停图的执行，并等待用户输入后再继续。开启支持人机协作的工作流程后，管理员可以查看意外情况并决定如何处理。中断操作需要一个检查点来保存暂停之间的状态。

In [1]:
from IPython.display import Image, display
import operator
from typing import Annotated, List, Literal, TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

In [2]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
config = {"configurable": {"thread_id": "1"}}

In [3]:
class State(TypedDict):
    nlist : Annotated[list[str], operator.add]  

In [4]:
def node_a(state: State) -> Command[Literal["b", "c", END]]:
    print("Entered 'a' node")
    select = state["nlist"][-1]
    if select == "b":
        next_node = "b"
    elif select == "c":
        next_node = "c"
    elif select == "q":
        next_node = END
    else:
        admin = interrupt(f"Unexpected input '{select}'")
        print(admin)
        if admin == "continue":
            next_node = "b"
        else:
            next_node = END
            select = "q"
            
    return Command(
        update = State(nlist = [select]),
        goto = next_node
    )


def node_b (state: State) -> State:
    return(State(nlist = ["B"]))

def node_c (state: State) -> State:
    return(State(nlist = ["C"]))

In [5]:
builder = StateGraph(State)

# Add nodes
builder.add_node("a", node_a)
builder.add_node("b", node_b)
builder.add_node("c", node_c)

# Add edges
builder.add_edge(START,"a")
builder.add_edge("b", END)
builder.add_edge("c", END)

# Compile
graph = builder.compile(checkpointer=memory)

In [6]:
while True:
    user = input('b, c, or q to quit: ')
    input_state = State(nlist = [user])
    result = graph.invoke(input_state, config)

    if '__interrupt__' in result:
        print(f"Interrupt:{result}")
        msg = result['__interrupt__'][-1].value
        print(msg)
        human = input(f"\n{msg}: ")

        human_response = Command(
            resume = human
        )
        result = graph.invoke(human_response, config)
        
    if result['nlist'][-1] == "q":
        print("quit")
        break


b, c, or q to quit:  b


Entered 'a' node


b, c, or q to quit:  c


Entered 'a' node


b, c, or q to quit:  d


Entered 'a' node
Interrupt:{'nlist': ['b', 'b', 'B', 'c', 'c', 'C', 'd'], '__interrupt__': [Interrupt(value="Unexpected input 'd'", id='e9fb113473bebd8564706f70a32758de')]}
Unexpected input 'd'



Unexpected input 'd':  continue


Entered 'a' node
continue


b, c, or q to quit:  q


Entered 'a' node
quit


<a id='l5_execution'></a>


## Takeaways

Setup:

- 节点 a 使用 `interrupt()` 函数在出现意外输入时暂停执行。
- 检查点器通过在暂停和恢复之间保存状态来启用中断。

Execution:

- 调用 `interrupt()` 时，流程图暂停并等待用户输入。
- 管理员的响应决定是继续执行还是结束执行。
- 流程图状态在暂停期间被保留，并在调用时被恢复。

Try Next:

- 修改中断逻辑，根据意外输入询问不同的问题。
- 在节点 b 或节点 c 中添加中断调用，以便在工作流程的不同点暂停。
